In [1]:
:set -XDeriveFunctor
import Data.List.NonEmpty (NonEmpty(..))
import Data.List (unfoldr)

-- Определим свой класс (аналог Control.Comonad из пакета comonad)
class Functor w => Comonad w where
  extract   :: w a -> a
  duplicate :: w a -> w (w a)
  duplicate = extend id
  extend    :: (w a -> b) -> w a -> w b
  extend f  = fmap f . duplicate

-- Удобный оператор (flip extend)
(=>>) :: Comonad w => w a -> (w a -> b) -> w b
(=>>) = flip extend
putStrLn "Setup done."

Setup done.

# 🫞 Комонады в Haskell — Полный справочник

> Комонада — это двойственная монада. Там где монада *вкладывает* значение в контекст и *сплющивает* слои,
> комонада *извлекает* значение из контекста и *расширяет* его вовне.

Каждая комонада описана по схеме:
- 📐 **Реализация** + интуиция «на пальцах»
- ⭐ **Главные операции**: `extract`, `duplicate`, `extend` (`=>>`)
- 🛠️ **Примеры** — от простого к сложному (минимум 3)
- 💡 **Ловушки** — контринтуитивные моменты
- 🔭 **Категорная точка зрения**

---
📌 Содержание

| # | Комонада | Суть |
|---|----------|------|
| 1 | `Identity` | Тривиальная комонада, законы |
| 2 | `Env e` / CoReader | Значение + фиксированное окружение |
| 3 | `Traced w` / CoWriter | Функция из моноида |
| 4 | `Store s` / CoState | Фокус + всё остальное (cellular automata!) |
| 5 | `Stream` | Бесконечный поток, sliding window |
| 6 | `NonEmpty` | Непустой список с фокусом |
| 7 | `Zipper` | Список с «курсором» |
| 8 | `Tree` | Дерево с фокусом |
| 9 | `CoFree f` | Свободная комонада |
| 10 | `Representable` | Представимые функторы |
| 11 | **🪞 Дуальность** | Монада ↔ Комонада, категорный взгляд, диаграммы |

---
> 💡 **Трансформеры комонад** (`EnvT`, `StoreT`, `TracedT`) — тема для отдельного ноутбука.

---
## 🧭 Класс типов `Comonad` — отправная точка

Прежде чем разбирать конкретные комонады, посмотрим на класс `Comonad` целиком.

```haskell
class Functor w => Comonad w where
  extract   :: w a -> a            -- "извлечь" фокус
  duplicate :: w a -> w (w a)      -- "обернуть каждую позицию в свой контекст"
  extend    :: (w a -> b) -> w a -> w b  -- применить "контекстную функцию"

  -- Минимально нужно реализовать: extract + (duplicate ИЛИ extend)
  extend f  = fmap f . duplicate
  duplicate = extend id
```

### Сравнение с монадой

```
  МОНАДА                         КОМОНАДА
  ──────────────────────────     ────────────────────────────
  return  :: a -> m a            extract   :: w a -> a
  join    :: m (m a) -> m a      duplicate :: w a -> w (w a)
  (>>=)   :: m a -> (a->m b)     extend    :: (w a->b) -> w a -> w b
           -> m b                (=>>)     :: w a -> (w a->b) -> w b
```

Стрелки **обращены**: монада *строит* контекст, комонада *разбирает* его.

### Законы комонады

```haskell
-- 1. extract . duplicate  ≡  id
-- 2. fmap extract . duplicate  ≡  id
-- 3. duplicate . duplicate  ≡  fmap duplicate . duplicate
```

Это **дуалы** монадических законов (левая единица, правая единица, ассоциативность).

---
# 1️⃣ Identity — «Тривиальная комонада»

## 📐 Реализация

`Identity` — это просто обёртка вокруг значения. Никакого «контекста» нет.
Она важна как **базовый случай** и для понимания законов.

```haskell
newtype Identity a = Identity { runIdentity :: a }

instance Comonad Identity where
  extract   (Identity x) = x
  duplicate (Identity x) = Identity (Identity x)
```

**Интуиция:** «контекст» состоит ровно из одного значения — самого `a`.
Нет ни соседей, ни истории, ни окружения. `extract` = `runIdentity`,
`duplicate` просто добавляет ещё один слой обёртки.

In [2]:
newtype MyIdentity a = MyIdentity { runMyIdentity :: a } deriving (Show, Functor, Eq)

instance Comonad MyIdentity where
  extract   (MyIdentity x) = x
  duplicate (MyIdentity x) = MyIdentity (MyIdentity x)

-- Проверим законы:
val :: MyIdentity Int
val = MyIdentity 42

-- Закон 1: extract . duplicate ≡ id
law1 :: Bool
law1 = (extract . duplicate) val == val

-- Закон 2: fmap extract . duplicate ≡ id
law2 :: Bool
law2 = (fmap extract . duplicate) val == val

-- Закон 3: duplicate . duplicate ≡ fmap duplicate . duplicate
law3 :: Bool
law3 = (runMyIdentity . runMyIdentity) ((duplicate . duplicate) val)
    == (runMyIdentity . runMyIdentity) ((fmap duplicate . duplicate) val)

putStrLn $ "Закон 1 (extract . duplicate = id): " ++ show law1
putStrLn $ "Закон 2 (fmap extract . duplicate = id): " ++ show law2
putStrLn $ "Закон 3 (ассоциативность duplicate): " ++ show law3

Закон 1 (extract . duplicate = id): True

Закон 2 (fmap extract . duplicate = id): True

Закон 3 (ассоциативность duplicate): True

## 🛠️ Примеры использования

`Identity` используют как «нейтральный трансформер» в стеках и для тестирования абстракций.

In [3]:
-- Пример 1: extend — применить функцию к "всему контексту"
-- Для MyIdentity "контекст" = само значение, поэтому extend f = fmap f
result1 :: MyIdentity String
result1 = extend (\ (MyIdentity n) -> "Значение: " ++ show n) (MyIdentity 42)
-- MyIdentity "Значение: 42"

-- Пример 2: =>> цепочка (ко-do)
result2 :: MyIdentity Int
result2 = MyIdentity 10
  =>> (\ (MyIdentity x) -> x * 2)   -- 20
  =>> (\ (MyIdentity x) -> x + 5)   -- 25

print result1
print result2

MyIdentity {runMyIdentity = "\1047\1085\1072\1095\1077\1085\1080\1077: 42"}

MyIdentity {runMyIdentity = 25}

## 💡 Ловушки

- `duplicate (Identity x) = Identity (Identity x)` — двойная обёртка, не `Identity x`!
  Это важно: `duplicate` всегда создаёт «контекст контекстов».
- Для `Identity` `extend f ≡ fmap (f . Identity . extract)` — всё сводится к обычному `fmap`.

## 🔭 Категорная точка зрения

`Identity` — **тождественный коэндофунктор** категории Haskell-типов.
В теории категорий комонада — это коалгебра над эндофунктором:
```
  ε : W → Id        (extract)
  δ : W → W∘W       (duplicate)
```
Для `Identity`: W = Id, поэтому `ε = id`, `δ = id` (в смысле изоморфизма).

---
# 2️⃣ Env e — «Значение с окружением» (CoReader)

## 📐 Реализация

`Env e a` хранит **пару**: некоторое окружение типа `e` и само значение `a`.

```haskell
data Env e a = Env e a   -- (окружение, значение)

instance Comonad (Env e) where
  extract   (Env _ x) = x          -- берём значение, игнорируем окружение
  duplicate (Env e x) = Env e (Env e x)  -- окружение "прорастает" вниз
```

**Интуиция:** представь конфигурационный файл (`e`) плюс текущий результат вычисления (`a`).
`extract` говорит «дай мне результат», `ask` (= `fst . runEnv`) говорит «дай мне конфиг».

**Дуальность с Reader:** `Reader r a ≅ r -> a`, `Env e a ≅ (e, a)`.
Монада Reader *потребляет* окружение (функция), комонада Env *несёт* его (пара).

In [4]:
data Env e a = Env e a deriving (Show, Functor)

instance Comonad (Env e) where
  extract   (Env _ x)   = x
  duplicate (Env e x)   = Env e (Env e x)

-- Вспомогательные функции
ask :: Env e a -> e
ask (Env e _) = e

asks :: (e -> b) -> Env e a -> b
asks f (Env e _) = f e

local :: (e -> e) -> Env e a -> Env e a
local f (Env e x) = Env (f e) x

runEnv :: Env e a -> (e, a)
runEnv (Env e x) = (e, x)

## 🛠️ Примеры использования

In [5]:
-- Пример 1: Вычисление с конфигурацией
data Config = Config { maxRetries :: Int, verbose :: Bool } deriving Show

computation :: Env Config Int
computation = Env (Config 3 True) 42

-- extend: функция может читать и конфиг, и текущее значение
withLogging :: Env Config Int -> String
withLogging env@(Env cfg val)
  | verbose cfg = "[LOG] result=" ++ show val ++ " (retries=" ++ show (maxRetries cfg) ++ ")"
  | otherwise   = show val

result :: Env Config String
result = extend withLogging computation

print result

Env (Config {maxRetries = 3, verbose = True}) "[LOG] result=42 (retries=3)"

In [6]:
-- Пример 2: Цепочка =>> с доступом к окружению
type Threshold = Int

scored :: Env Threshold Double
scored = Env 50 73.5

-- Каждый шаг видит и порог, и текущее значение
graded :: Env Threshold String
graded = scored
  =>> (\ env@(Env thr val) -> if val >= fromIntegral thr then "Pass" else "Fail")
  =>> (\ (Env thr grade) -> grade ++ " (threshold=" ++ show thr ++ ")")

print graded

Env 50 "Pass (threshold=50)"

In [7]:
-- Пример 3: local — временно изменить окружение
strictMode :: Env Config Int -> Env Config Int
strictMode = local (\ cfg -> cfg { maxRetries = 0 })

original  = Env (Config 3 True) 100
strict    = strictMode original

putStrLn $ "Original config: " ++ show (ask original)
putStrLn $ "Strict config:   " ++ show (ask strict)
putStrLn $ "Value unchanged: " ++ show (extract strict)

Original config: Config {maxRetries = 3, verbose = True}

Strict config:   Config {maxRetries = 0, verbose = True}

Value unchanged: 100

## 💡 Ловушки

- **`duplicate` копирует окружение**, а не значение: `Env e (Env e x)` — оба слоя
  разделяют *одно* `e`. Это важно: при `extend` все «позиции» видят одно окружение.
- Нельзя «изменить» окружение через `extend` — только через `local` (или явный паттерн-матчинг).
  Это отличие от монады `State`, где состояние *течёт* через вычисления.
- **`Env e a` — НЕ монада** (за исключением тривиальных случаев). Нет способа «вложить» новое
  окружение внутрь, не разрушив структуру.

## 🔭 Категорная точка зрения

`Env e` — это **комонада произведения** (product comonad):

```
  W = (e, ─) : Set → Set
  ε (e, a)     = a                   (проекция на второй компонент)
  δ (e, a)     = (e, (e, a))         (диагональ: e "разветвляется"  )
```

Она возникает из **сопряжения** (adjunction) `─ × e ⊣ (─)^e`:
```
  Env e  ←→  Reader e
  левый сопряжённый  ←→  правый сопряжённый
```
Из каждого сопряжения `F ⊣ G` возникает монада `GF` и комонада `FG`.
Здесь `F = ─ × e`, `G = e → ─`, поэтому монада = Reader, комонада = Env.

---
# 3️⃣ Traced w — «Функция из моноида» (CoWriter)

## 📐 Реализация

`Traced w a` — это **функция** `w -> a`, где `w` — моноид.

```haskell
newtype Traced w a = Traced { runTraced :: w -> a }

instance Monoid w => Comonad (Traced w) where
  extract   (Traced f)   = f mempty          -- вычислить при "нулевом" входе
  duplicate (Traced f)   = Traced (\ w -> Traced (\ w' -> f (w <> w')))
```

**Интуиция:** представь функцию "посмотреть значение в точке `w`".
- `extract` = «что в точке 0 (нейтральном элементе)?»
- `duplicate` = «создать функцию, которая добавляет смещение к уже имеющемуся»
- `extend f t` = «создать новую функцию, вычисляя `f` в каждой "сдвинутой" версии `t`»

**Дуальность с Writer:** `Writer w a ≅ (a, w)`, `Traced w a ≅ w -> a`.
Writer *аккумулирует* моноид, Traced *индексируется* моноидом.

In [8]:
newtype Traced w a = Traced { runTraced :: w -> a } deriving Functor

instance Monoid w => Comonad (Traced w) where
  extract   (Traced f)   = f mempty
  duplicate (Traced f)   = Traced (\ w -> Traced (\ w' -> f (w <> w')))

-- Вспомогательные функции
trace :: Monoid w => w -> Traced w a -> a
trace w (Traced f) = f w

-- "сдвинуть" вычисление на w
listen :: Monoid w => Traced w a -> Traced w (a, w)
listen (Traced f) = Traced (\ w -> (f w, w))

## 🛠️ Примеры использования

In [9]:
import Data.Monoid (Sum(..))

-- Пример 1: Traced Sum Int — "вычисление в точке n"
-- Сумма квадратов от 0 до n

sumSquares :: Traced (Sum Int) Int
sumSquares = Traced (\ (Sum n) -> sum [i^2 | i <- [0..n]])

-- extract = значение в точке mempty = Sum 0
putStrLn $ "В точке 0: " ++ show (extract sumSquares)       -- 0
putStrLn $ "В точке 3: " ++ show (trace (Sum 3) sumSquares) -- 0+1+4+9=14
putStrLn $ "В точке 5: " ++ show (trace (Sum 5) sumSquares) -- 55

В точке 0: 0

В точке 3: 14

В точке 5: 55

In [10]:
-- Пример 2: extend — создать производную функцию
-- "Разница между точкой w+1 и точкой w" (дискретная производная)

import Data.Monoid (Sum(..))

derivative :: Traced (Sum Int) Int -> Int
derivative t = trace (Sum 1) t - extract t

-- Исходная функция: n^2
squares :: Traced (Sum Int) Int
squares = Traced (\ (Sum n) -> n * n)

-- extend derivative: вычислить производную в каждой точке
derivSquares :: Traced (Sum Int) Int
derivSquares = extend derivative squares
-- derivSquares в точке n = (n+1)^2 - n^2 = 2n+1

putStrLn "Функция n^2 и её дискретная производная 2n+1:"
mapM_ (\ n -> putStrLn $ "n=" ++ show n
                        ++ "  f=" ++ show (trace (Sum n) squares)
                        ++ "  f'=" ++ show (trace (Sum n) derivSquares))
      [0..5]

Функция n^2 и её дискретная производная 2n+1:

n=0  f=0  f'=1
n=1  f=1  f'=3
n=2  f=4  f'=5
n=3  f=9  f'=7
n=4  f=16  f'=9
n=5  f=25  f'=11

In [11]:
-- Пример 3: Traced [String] — "журнал шагов"
-- Моноид = [String] (список шагов), значение = описание пути

type Log = [String]

pathDescription :: Traced Log String
pathDescription = Traced (\ steps ->
  case steps of
    []  -> "Начало"
    _   -> "Начало -> " ++ foldl1 (\ a b -> a ++ " -> " ++ b) steps)

-- Посмотреть путь через определённые шаги
putStrLn $ trace []                          pathDescription
putStrLn $ trace ["A"]                       pathDescription
putStrLn $ trace ["A","B","C"]               pathDescription

-- extend: в каждой "позиции" добавляем ещё один шаг
withExtra :: Traced Log String -> String
withExtra t = trace ["X"] t ++ " [+X]"

extended :: Traced Log String
extended = extend withExtra pathDescription
putStrLn $ trace ["A","B"] extended

Начало

Начало -> A

Начало -> A -> B -> C

Начало -> A -> B -> X [+X]

## 💡 Ловушки

- **`extract` = значение при `mempty`**, не при каком-то «первом» элементе.
  Для `Sum Int` это `f 0`, для `[a]` это `f []`.
- **`duplicate` добавляет смещение**, а не заменяет: `runTraced (duplicate t) w w' = runTraced t (w <> w')`.
  Внешний слой «накапливает», внутренний «добавляет ещё».
- Для `Traced` нет прямого аналога `local` (Env). Чтобы «сдвинуть фокус», используют `trace w`.
- `Traced w a` требует `Monoid w` — без моноида нет `extract` (нет `mempty`).

## 🔭 Категорная точка зрения

`Traced w` — **комонада функций из моноида**:

```
  W = (w →) : Set → Set    (контравариантно по w, ковариантно по a)
  ε f       = f ε           (вычислить в нейтральном элементе)
  δ f       = λw. λw'. f (w ⊗ w')   (умножение в моноиде)
```

Возникает из **сопряжения** `w × ─ ⊣ w → ─` (то же что Env!):
- `Env w` = комонада `(w × ─) ∘ (w → ─) = w × (w → ─)`
- `Traced w` = монада... подождите, нет: `(w → ─) ∘ (w × ─) = w → (w × ─)`

На самом деле `Traced w` — **другая** комонада из того же сопряжения,
если поменять порядок композиции функторов:
```
  F ⊣ G  →  GF = монада,  FG = комонада
  Здесь F = w×─, G = w→─
  GF = w→(w×─) = Traced-подобное  (на самом деле это Reader!)
  FG = w×(w→─) = Env
```
`Traced w` = комонада из сопряжения `Free w ⊣ Forget` над категорией моноидов.

---
# 4️⃣ Store s — «Фокус в хранилище» (CoState)

## 📐 Реализация

`Store s a` хранит **функцию** `s -> a` (всё "хранилище") и **текущую позицию** `s`.

```haskell
data Store s a = Store (s -> a) s

instance Comonad (Store s) where
  extract   (Store f s)   = f s              -- значение в текущей позиции
  duplicate (Store f s)   = Store (Store f) s  -- каждая позиция = свой Store
```

**Интуиция:** это как массив с курсором.
- `extract` = «прочитай текущий элемент»
- `pos` = «где курсор?»
- `peek i` = «прочитай элемент в позиции `i`» (не двигая курсор)
- `duplicate` = «в каждой позиции `i` — Store, сфокусированный на `i`»
- `extend f` = «создай новый массив, где каждый элемент — `f` применённая к Store-у сфокусированному на этой позиции»

**Дуальность с State:** `State s a ≅ s -> (a, s)`, `Store s a ≅ (s -> a, s)`.

In [12]:
data Store s a = Store (s -> a) s

instance Functor (Store s) where
  fmap f (Store g s) = Store (f . g) s

instance Comonad (Store s) where
  extract   (Store f s) = f s
  duplicate (Store f s) = Store (Store f) s

-- Вспомогательные функции
pos :: Store s a -> s
pos (Store _ s) = s

peek :: s -> Store s a -> a
peek s' (Store f _) = f s'

peeks :: (s -> s) -> Store s a -> a
peeks f (Store g s) = g (f s)

seek :: s -> Store s a -> Store s a
seek s' (Store f _) = Store f s'

seeks :: (s -> s) -> Store s a -> Store s a
seeks f (Store g s) = Store g (f s)

experiment :: Functor f => (s -> f s) -> Store s a -> f a
experiment f (Store g s) = fmap g (f s)

## 🛠️ Примеры использования

In [13]:
-- Пример 1: Простой массив с курсором
type Grid1D = Store Int Int

-- "Массив" = функция Int->Int, курсор на позиции 5
arr :: Grid1D
arr = Store (\ i -> i * i) 5   -- arr[i] = i^2, фокус на 5

putStrLn $ "Текущее значение (pos=5): " ++ show (extract arr)     -- 25
putStrLn $ "Значение в pos=3: "         ++ show (peek 3 arr)       -- 9
putStrLn $ "Значение в pos=7: "         ++ show (peek 7 arr)       -- 49
putStrLn $ "Текущая позиция: "          ++ show (pos arr)          -- 5

Текущее значение (pos=5): 25

Значение в pos=3: 9

Значение в pos=7: 49

Текущая позиция: 5

In [14]:
-- Пример 2: extend — "свёрточный фильтр" (moving average)
-- Создаём новый "массив", где каждый элемент = среднее трёх соседей

movingAvg :: Store Int Double -> Double
movingAvg store =
  let i = pos store
      vals = map (\ d -> peek (i+d) store) [-1, 0, 1]
  in  sum vals / 3.0

-- Исходный "массив": sin(i/2)
signal :: Store Int Double
signal = Store (\ i -> sin (fromIntegral i / 2.0)) 0

smoothed :: Store Int Double
smoothed = extend movingAvg signal

putStrLn "Исходный сигнал vs сглаженный (позиции 0..5):"
mapM_ (\ i -> putStrLn $ "pos=" ++ show i
                         ++ "  orig=" ++ show (peek i signal)
                         ++ "  smooth=" ++ show (peek i smoothed))
      [0..5]

Исходный сигнал vs сглаженный (позиции 0..5):

pos=0  orig=0.0  smooth=0.0
pos=1  orig=0.479425538604203  smooth=0.4402988411373665
pos=2  orig=0.8414709848078965  smooth=0.7727971700053846
pos=3  orig=0.9974949866040544  smooth=0.9160877994125443
pos=4  orig=0.9092974268256817  smooth=0.8350881858445641
pos=5  orig=0.5984721441039565  smooth=0.5496298596631685

In [15]:
-- Пример 3: Клеточный автомат Rule 110 (1D)
-- Store Bool — лента булевых значений, фокус = текущая клетка

type Cell   = Bool
type Tape   = Store Int Cell

-- Начальное состояние: только клетка 0 = True
initTape :: Tape
initTape = Store (\ i -> i == 0) 0

-- Правило Rule 110: следующее поколение клетки зависит от трёх соседей
rule110 :: Tape -> Cell
rule110 t =
  let l = peek (pos t - 1) t
      c = extract t
      r = peek (pos t + 1) t
  in case (l, c, r) of
    (True,  True,  True)  -> False
    (True,  True,  False) -> True
    (True,  False, True)  -> True
    (True,  False, False) -> False
    (False, True,  True)  -> True
    (False, True,  False) -> True
    (False, False, True)  -> True
    (False, False, False) -> False

-- Следующее поколение = extend rule110
step :: Tape -> Tape
step = extend rule110

-- Отобразить несколько поколений (позиции -10..10)
showTape :: Tape -> String
showTape t = map (\ i -> if peek i t then '#' else '.') [-10..10]

-- Прогоним 8 поколений
let generations = iterate step initTape
mapM_ (putStrLn . showTape) (take 8 generations)

..........#..........
.........##..........
........###..........
.......##.#..........
......#####..........
.....##...#..........
....###..##..........
...##.#.###..........

In [16]:
-- Пример 4: 2D клеточный автомат — Игра Жизни Конвея
type Pos2D  = (Int, Int)
type Grid2D = Store Pos2D Bool

-- Начальное состояние: планер
glider :: Grid2D
glider = Store alive (0,0)
  where
    aliveCells = [(0,1),(1,2),(2,0),(2,1),(2,2)]
    alive p    = p `elem` aliveCells

-- Количество живых соседей
neighbors :: Grid2D -> Int
neighbors g =
  let (x, y) = pos g
      offsets = [(dx,dy) | dx <- [-1..1], dy <- [-1..1], (dx,dy) /= (0,0)]
  in  length . filter id $ map (\ (dx,dy) -> peek (x+dx, y+dy) g) offsets

-- Правило Конвея
conway :: Grid2D -> Bool
conway g =
  let alive = extract g
      n     = neighbors g
  in  if alive then n == 2 || n == 3
               else n == 3

stepConway :: Grid2D -> Grid2D
stepConway = extend conway

showGrid :: Grid2D -> String
showGrid g = unlines
  [ concat [ if peek (x,y) g then "█ " else "· " | x <- [-1..4] ]
  | y <- [-1..4] ]

putStrLn "Поколение 0 (планер):"
putStrLn (showGrid glider)
putStrLn "Поколение 4:"
putStrLn (showGrid (iterate stepConway glider !! 4))

Поколение 0 (планер):

· · · · · · 
· · · █ · · 
· █ · █ · · 
· · █ █ · · 
· · · · · · 
· · · · · ·

Поколение 4:

· · · · · · 
· · · · · · 
· · · · █ · 
· · █ · █ · 
· · · █ █ · 
· · · · · ·

## 💡 Ловушки

- **`duplicate` — не `fmap (Store f)`**: `duplicate (Store f s) = Store (Store f) s`.
  Внутренняя функция — это `Store f`, которая при вызове с `s'` даёт `Store f s'`.
  То есть `peek s' (extract (duplicate st)) = peek s' st` — каждый «внутренний» Store
  смотрит на то же хранилище, только с другим курсором.
- **`extend f` — это свёртка**: новый Store в позиции `s'` содержит
  `f (Store arr s')`. Это именно "применить f к каждому сдвинутому контексту".
- **Бесконечные структуры**: Store неявно описывает *бесконечный* массив.
  `fmap` и `extend` работают ленivo — нет проблем с производительностью.

## 🔭 Категорная точка зрения

`Store s` — **комонада состояния** (state comonad):

```
  Store s a  ≅  (s → a) × s
  ε (f, s)   = f s              (вычислить в точке)
  δ (f, s)   = (λs'. (f, s'), s)  (каждая позиция несёт полное хранилище)
```

Возникает из сопряжения **диагонали и произведения**:
```
  (─ × s) ⊣ (─)^s    (то же что для Env/Reader!)
  Но теперь:  F = ─×s,  G = s→─
  FG = (s→─)×s = Store s      ← комонада
  GF = s→(─×s) = State s      ← монада
```

Это **самое важное сопряжение** в функциональном программировании.

---
# 5️⃣ Stream — «Бесконечный поток»

## 📐 Реализация

`Stream a` — бесконечный список: голова и хвост (тоже `Stream`).

```haskell
data Stream a = a :> Stream a   -- всегда бесконечный!

instance Comonad Stream where
  extract   (x :> _)  = x           -- голова = "текущий" элемент
  duplicate s@(_ :> t) = s :> duplicate t  -- каждый хвост = свой Stream
```

**Интуиция:** это бесконечная лента (как лента Тьюринга).
- `extract` = «прочитай текущий символ»
- `duplicate` = «создай Stream из всех суффиксов»: `[s0,s1,s2,...], [s1,s2,...], ...`
- `extend f` = «создай новый Stream, где `i`-й элемент = `f` применённая к хвосту начиная с `i`»

Это **точно** sliding window по бесконечной последовательности!

In [17]:
data Stream a = a :> Stream a

infixr 5 :>

instance Functor Stream where
  fmap f (x :> xs) = f x :> fmap f xs

instance Comonad Stream where
  extract   (x :> _)   = x
  duplicate s@(_ :> t) = s :> duplicate t

-- Вспомогательные функции
fromList :: [a] -> Stream a   -- бесконечное повторение
fromList xs = go (cycle xs) where go (y:ys) = y :> go ys

fromListWith :: a -> [a] -> Stream a  -- с дефолтным значением
fromListWith def [] = let inf = def :> inf in inf
fromListWith def (x:xs) = x :> fromListWith def xs

toList :: Int -> Stream a -> [a]
toList 0 _        = []
toList n (x :> xs) = x : toList (n-1) xs

-- Натуральные числа
nats :: Stream Int
nats = go 0 where go n = n :> go (n+1)

-- Числа Фибоначчи
fibs :: Stream Integer
fibs = go 0 1 where go a b = a :> go b (a+b)

Line 17: Use foldr
Found:
fromListWith def [] = let inf = def :> inf in inf
fromListWith def (x : xs) = x :> fromListWith def xs
Why not:
fromListWith def xs = foldr (:>) (let inf = def :> inf in inf) xs

## 🛠️ Примеры использования

In [18]:
-- Пример 1: Базовые операции
putStrLn $ "Первые 10 натуральных: " ++ show (toList 10 nats)
putStrLn $ "Первые 10 Фибоначчи:   " ++ show (toList 10 fibs)

-- extend: создаём новый Stream где каждый элемент = сумма текущего и следующего
pairwiseSum :: Stream Int -> Int
pairwiseSum (x :> y :> _) = x + y

sumPairs :: Stream Int
sumPairs = extend pairwiseSum nats
putStrLn $ "Суммы пар (0+1, 1+2, ...): " ++ show (toList 8 sumPairs)

Первые 10 натуральных: [0,1,2,3,4,5,6,7,8,9]

Первые 10 Фибоначчи:   [0,1,1,2,3,5,8,13,21,34]

Суммы пар (0+1, 1+2, ...): [1,3,5,7,9,11,13,15]

In [19]:
-- Пример 2: Sliding window — скользящее окно
slidingWindow :: Int -> Stream a -> Stream [a]
slidingWindow n = extend (toList n)

-- Скользящее среднее
movingMean :: Stream Double -> Double
movingMean s = let window = 5
                   vals   = toList window s
               in  sum vals / fromIntegral window

signalS :: Stream Double
signalS = fmap (\ n -> sin (fromIntegral n * 0.5)) nats

smoothedS :: Stream Double
smoothedS = extend movingMean signalS

putStrLn "Окна по 3 элемента из [0..9]:"
print $ toList 8 (slidingWindow 3 (fmap fromIntegral nats))

putStrLn "Сглаженный сигнал (первые 6):"
print $ map (\ x -> fromIntegral (round (x * 100)) / 100.0) (toList 6 smoothedS)

Окна по 3 элемента из [0..9]:

[[0,1,2],[1,2,3],[2,3,4],[3,4,5],[4,5,6],[5,6,7],[6,7,8],[7,8,9]]

Сглаженный сигнал (первые 6):

[0.65,0.77,0.7,0.46,0.11,-0.27]

In [20]:
-- Пример 3: Кодирование/декодирование потоков (Run-Length Encoding интуиция)
-- duplicate даёт Stream суффиксов — полезно для look-ahead парсинга

-- Посмотрим на duplicate: Stream суффиксов
suffixes :: Stream (Stream Int)
suffixes = duplicate (fmap fromIntegral nats)

putStrLn "duplicate nats — первые 3 суффикса (по 5 эл-тов каждый):"
let ss = toList 3 suffixes
mapM_ (\ s -> print (toList 5 s)) ss

duplicate nats — первые 3 суффикса (по 5 эл-тов каждый):

[0,1,2,3,4]
[1,2,3,4,5]
[2,3,4,5,6]

In [21]:
-- Пример 4: Правило 90 — клеточный автомат на Stream
-- Правило 90: XOR соседей

rule90 :: Stream Bool -> Bool
rule90 (l :> c :> r :> _) = l /= r  -- XOR соседей
-- Но нам нужен доступ к левому соседу! Stream смотрит только вперёд.
-- Поэтому используем Store для 1D CA (см. секцию 4).
-- Для Stream удобнее right-only automata:

-- Правило Фибоначчи: следующий элемент = XOR двух предыдущих (Xorshift-подобно)
xorNext :: Stream Bool -> Bool
xorNext (a :> b :> _) = a /= b

boolStream :: Stream Bool
boolStream = True :> False :> extend xorNext boolStream  -- ленивая рекурсия невозможна через extend, используем явно
-- Вместо этого — явная рекурсия:
boolSeq :: Stream Bool
boolSeq = go True False
  where go a b = a :> go b (a /= b)

putStrLn "XOR-последовательность (первые 12):"
print $ map (\ b -> if b then '1' else '0') (toList 12 boolSeq)

XOR-последовательность (первые 12):

"101101101101"

## 💡 Ловушки

- **Stream всегда бесконечен**: нет конструктора пустого потока. Это хорошо — гарантирует
  тотальность `extract` и `duplicate`.
- **`duplicate` = все суффиксы**: `toList k (duplicate s) = [drop 0 s, drop 1 s, ..., drop (k-1) s]`.
  Это значит: `extend f s` на позиции `i` = `f (drop i s)`.
- **Левые соседи недоступны**: Stream смотрит только вперёд. Для двунаправленного доступа
  используйте `Zipper` (секция 7) или `Store Int` (секция 4).
- **Лень критична**: благодаря ленивым вычислениям `extend` и `duplicate` работают за O(1)
  на шаг, несмотря на «бесконечность».

## 🔭 Категорная точка зрения

`Stream` — это **конечная коалгебра** функтора `F a = Int × a` (коиндуктивный тип):
```
  unfold : (s → Int × s) → s → Stream Int
  ε (x :> _) = x     (голова = проекция)
  δ s        = s :> δ (tail s)  (разворачивание)
```
`Stream` — терминальная коалгебра функтора `F X = A × X`:
любая коалгебра `(S, f : S → A×S)` однозначно факторизуется через `Stream A`.
Это делает `Stream` «наименьшей» комонадой над потоковым функтором.

---
# 6️⃣ NonEmpty — «Непустой список с фокусом»

## 📐 Реализация

`NonEmpty a` — список с гарантированной головой. Фокус = голова.

```haskell
data NonEmpty a = a :| [a]

instance Comonad NonEmpty where
  extract   (x :| _)  = x    -- голова
  duplicate xs@(_ :| t) = xs :| case t of
    []     -> []
    (y:ys) -> toList (duplicate (y :| ys))
```

**Интуиция:** это как `Stream`, но конечный. Фокус = текущий элемент (голова).
`duplicate` = «все суффиксы»: `[1,2,3] → [[1,2,3],[2,3],[3]]`.

Ключевое отличие от `[]`: `extract` тотален — всегда есть хотя бы один элемент.

In [22]:
import Data.List.NonEmpty (NonEmpty(..))
import qualified Data.List.NonEmpty as NE

instance Comonad NonEmpty where
  extract (x :| _) = x
  duplicate xs@(x :| rest) =
    xs :| case rest of
      []     -> []
      (y:ys) -> NE.toList (duplicate (y :| ys))

-- Пример: суффиксы
example :: NonEmpty Int
example = 1 :| [2, 3, 4, 5]

duplicated :: NonEmpty (NonEmpty Int)
duplicated = duplicate example

putStrLn "Все суффиксы [1,2,3,4,5]:"
mapM_ (print . NE.toList) duplicated

Все суффиксы [1,2,3,4,5]:

[1,2,3,4,5]
[2,3,4,5]
[3,4,5]
[4,5]
[5]

## 🛠️ Примеры использования

In [23]:
-- Пример 1: extend — вычислить что-то для каждого суффикса
-- "суммировать хвост от каждой позиции"
suffixSum :: NonEmpty Int -> Int
suffixSum = sum . NE.toList

sums :: NonEmpty Int
sums = extend suffixSum example
-- [1+2+3+4+5, 2+3+4+5, 3+4+5, 4+5, 5] = [15,14,12,9,5]
putStrLn $ "Суффиксные суммы: " ++ show (NE.toList sums)

Суффиксные суммы: [15,14,12,9,5]

In [24]:
-- Пример 2: running maximum (максимум от текущей позиции до конца)
runMax :: NonEmpty Int -> Int
runMax = maximum . NE.toList

maxSuffix :: NonEmpty Int
maxSuffix = extend runMax (3 :| [1, 4, 1, 5, 9, 2, 6])
putStrLn $ "Максимум суффикса: " ++ show (NE.toList maxSuffix)

Максимум суффикса: [9,9,9,9,9,9,6,6]

In [25]:
-- Пример 3: контекстный анализ текста
-- Каждое слово + все следующие слова
words' :: NonEmpty String
words' = "Haskell" :| ["is", "purely", "functional"]

-- Для каждой позиции: слово + количество слов после него
context :: NonEmpty String -> String
context (w :| rest) = w ++ " (" ++ show (length rest) ++ " words follow)"

analysis :: NonEmpty String
analysis = extend context words'
mapM_ putStrLn analysis

Haskell (3 words follow)
is (2 words follow)
purely (1 words follow)
functional (0 words follow)

## 💡 Ловушки

- **`duplicate` = суффиксы**, не все подсписки и не все подмножества.
- `NonEmpty` из `Data.List.NonEmpty` уже имеет инстанс `Comonad` в пакете `comonad`.
  Наш инстанс выше — демонстрационный.
- Для «двунаправленного» NonEmpty (нужны и предыдущие элементы) используйте `Zipper`.

## 🔭 Категорная точка зрения

`NonEmpty` — **коалгебра «непустого списка»**:
```
  F X = A + A×X    (база коиндуктивного типа)
  NonEmpty A = νX. A + A×X
```
Это **финальная коалгебра** (как Stream, но конечная).
`extract` = `ε` = проекция на голову. `duplicate` = unfold суффиксов.

---
# 7️⃣ Zipper — «Список с курсором»

## 📐 Реализация

`Zipper a` — список, «разрезанный» в точке фокуса: левая часть (в обратном порядке),
текущий элемент, правая часть.

```haskell
data Zipper a = Zipper [a] a [a]
--               левые  фокус  правые
--             (обратный порядок)
```

**Интуиция:** курсор текстового редактора. Слева от курсора — уже написанное (в обратном порядке),
справа — ещё не написанное. `extract` = «символ под курсором».

**Ключевое**: `duplicate` создаёт Zipper из всех *позиций* — в каждой позиции стоит
Zipper, сфокусированный на этой позиции. Это позволяет `extend` видеть
**весь контекст вокруг каждого элемента**.

In [26]:
data Zipper a = Zipper [a] a [a] deriving Show

instance Functor Zipper where
  fmap f (Zipper l c r) = Zipper (map f l) (f c) (map f r)

instance Comonad Zipper where
  extract (Zipper _ c _) = c

  duplicate z@(Zipper l c r) =
    Zipper lefts z rights
    where
      -- конечные сдвиги: останавливаемся когда достигли края
      lefts  = unfoldr stepLeft  z
      rights = unfoldr stepRight z
      stepLeft  (Zipper []     _ _) = Nothing
      stepLeft  zz                  = let zz' = goLeft zz  in Just (zz', zz')
      stepRight (Zipper _ _ [])     = Nothing
      stepRight zz                  = let zz' = goRight zz in Just (zz', zz')

-- Навигация
goLeft :: Zipper a -> Zipper a
goLeft  (Zipper (l:ls) c r) = Zipper ls     l (c:r)
goLeft  z@(Zipper [] _ _)   = z  -- край

goRight :: Zipper a -> Zipper a
goRight (Zipper l c (r:rs)) = Zipper (c:l)  r rs
goRight z@(Zipper _ _ [])   = z  -- край

-- Создать Zipper из списка
fromList' :: [a] -> Maybe (Zipper a)
fromList' []     = Nothing
fromList' (x:xs) = Just (Zipper [] x xs)

-- Все элементы Zipper в порядке
toList' :: Zipper a -> [a]
toList' (Zipper l c r) = reverse l ++ [c] ++ r

## 🛠️ Примеры использования

In [27]:
-- Пример 1: Базовые операции
let Just z = fromList' [1..7 :: Int]
z2 = goRight (goRight z)  -- сместились на 2 вправо

putStrLn $ "Начальный Zipper: " ++ show (toList' z)
putStrLn $ "После двух goRight: " ++ show z2
putStrLn $ "Фокус: " ++ show (extract z2)  -- 3
putStrLn $ "Весь список: " ++ show (toList' z2)

Начальный Zipper: [1,2,3,4,5,6,7]

После двух goRight: Zipper [2,1] 3 [4,5,6,7]

Фокус: 3

Весь список: [1,2,3,4,5,6,7]

In [28]:
-- Пример 2: extend — свёртка с доступом к соседям
-- Каждый элемент заменяем суммой его и двух соседей

neighborhood :: Zipper Int -> Int
neighborhood (Zipper l c r) =
  let left1  = take 1 l   -- 1 сосед слева
      right1 = take 1 r   -- 1 сосед справа
  in  sum (left1 ++ [c] ++ right1)

let Just nums = fromList' [1..7 :: Int]
let blurred   = extend neighborhood nums
putStrLn $ "Исходный:    " ++ show (toList' nums)
putStrLn $ "Сглаженный:  " ++ show (toList' blurred)

Исходный:    [1,2,3,4,5,6,7]

Сглаженный:  [3,6,9,12,15,18,13]

In [29]:
-- Пример 3: Редактор строки — операции с текстом
type TextEditor = Zipper Char

mkEditor :: String -> Maybe TextEditor
mkEditor = fromList'

insertChar :: Char -> TextEditor -> TextEditor
insertChar c (Zipper l cur r) = Zipper (cur:l) c r

deleteForward :: TextEditor -> TextEditor
deleteForward z@(Zipper _ _ [])    = z
deleteForward   (Zipper l c (_:r)) = Zipper l c r  -- удалить справа

showCursor :: TextEditor -> String
showCursor (Zipper l c r) =
  reverse l ++ "[" ++ [c] ++ "]" ++ r

let Just ed = mkEditor "Hello World"
    ed2 = goRight (goRight (goRight (goRight (goRight ed))))  -- cursor на 'W'
putStrLn $ "Редактор: " ++ showCursor ed2

Редактор: Hello[ ]World

In [30]:
-- Пример 4: duplicate — "все виды" на список
let Just small = fromList' [1..4 :: Int]
let allViews = duplicate small

putStrLn "duplicate — Zipper всех Zipper-ов (фокус в каждой позиции):"
let allZ = toList' allViews
mapM_ (\ z -> putStrLn $ "  фокус=" ++ show (extract z) ++ "  контекст=" ++ show (toList' z)) allZ

duplicate — Zipper всех Zipper-ов (фокус в каждой позиции):

  фокус=1  контекст=[1,2,3,4]
  фокус=2  контекст=[1,2,3,4]
  фокус=3  контекст=[1,2,3,4]
  фокус=4  контекст=[1,2,3,4]

## 💡 Ловушки

- **Левая часть хранится в обратном порядке** — это для O(1) перемещения курсора.
  `toList'` должен делать `reverse l`.
- **`duplicate` использует `iterate`** с `goLeft`/`goRight` — это создаёт бесконечные
  (ленивые) хвосты, обрезанные краями. На краях `goLeft`/`goRight` = `id`.
- **Zipper — деривация по типу**: это буквально производная типа `[a]` по `a`!
  `d/da ([a]) = [a] × [a]` — пара из левого и правого контекстов.
- `extend f` на Zipper = применить `f` к каждой «позиционной версии» — это мощнее,
  чем `fmap`, потому что `f` видит **весь список**, а не один элемент.

## 🔭 Категорная точка зрения

Zipper — это **производная типа** в смысле Конора МакБрайда:
```
  ∂/∂A ([A]) = [A] × [A]   (левый контекст × правый контекст)
```
Более общо: «дыра в типе `F`» = `∂F/∂A`, и эта дыра + значение = Zipper.
```
  Zipper F a = (∂F/∂a) × a
  Comonad Zipper ← существование для дифференцируемых функторов
```
Это обобщается: **любой дифференцируемый функтор порождает комонаду Zipper**.

---
# 8️⃣ Tree — «Розовое дерево с фокусом»

## 📐 Реализация

`Tree a` (Rose Tree) — дерево с меткой в каждом узле и произвольным числом детей.

```haskell
data Tree a = Node { rootLabel :: a, subForest :: [Tree a] }

instance Comonad Tree where
  extract   = rootLabel
  duplicate t@(Node _ ts) = Node t (map duplicate ts)
```

**Интуиция:** дерево с фокусом в корне.
- `extract` = метка корня (= «текущая» вершина)
- `duplicate` = каждый узел заменяется **поддеревом с корнем в этом узле**
- `extend f t` = в каждый узел записать `f (поддерево с корнем в этом узле)`

Это мощная операция: `f` видит всё поддерево, а не только метку узла.

In [31]:
data Tree a = Node { rootLabel :: a, subForest :: [Tree a] } deriving (Show, Eq)

instance Functor Tree where
  fmap f (Node x ts) = Node (f x) (map (fmap f) ts)

instance Comonad Tree where
  extract   (Node x _)  = x
  duplicate t@(Node _ ts) = Node t (map duplicate ts)

-- Вспомогательные
leaf :: a -> Tree a
leaf x = Node x []

depth :: Tree a -> Int
depth (Node _ []) = 0
depth (Node _ ts) = 1 + maximum (map depth ts)

size :: Tree a -> Int
size (Node _ ts) = 1 + sum (map size ts)

showTree :: Show a => Tree a -> String
showTree = go 0
  where
    go indent (Node x ts) =
      replicate (indent*2) ' ' ++ show x ++ "\n"
      ++ concatMap (go (indent+1)) ts

## 🛠️ Примеры использования

In [32]:
-- Пример 1: Базовое дерево и duplicate
sampleTree :: Tree Int
sampleTree = Node 1
  [ Node 2 [leaf 4, leaf 5]
  , Node 3 [leaf 6]
  ]

putStrLn "Исходное дерево:"
putStr (showTree sampleTree)
putStrLn $ "\nРазмер: " ++ show (size sampleTree)
putStrLn $ "Глубина: " ++ show (depth sampleTree)

-- duplicate: каждый узел = поддерево
let duplTree = duplicate sampleTree
putStrLn "\nduplicate: в корне стоит всё дерево"
putStrLn $ "extract (duplicate t) = t? " ++ show (extract duplTree == sampleTree)

Исходное дерево:

1
  2
    4
    5
  3
    6


Размер: 6

Глубина: 2


duplicate: в корне стоит всё дерево

extract (duplicate t) = t? True

In [33]:
-- Пример 2: extend — аннотировать каждый узел размером его поддерева
annotateSize :: Tree Int -> Tree Int
annotateSize = extend size

sized :: Tree Int
sized = annotateSize sampleTree

putStrLn "Дерево, аннотированное размером поддерева:"
putStr (showTree sized)
-- корень = 6 (всё дерево), Node 2 = 3 (сам + 2 листа), листья = 1

Дерево, аннотированное размером поддерева:

6
  3
    1
    1
  2
    1

In [34]:
-- Пример 3: Вычисление глубины каждого узла
-- Хитрость: depth считает глубину ПОДДЕРЕВА, нам нужна глубина узла от корня.
-- Используем annotate с передачей уровня через State-подобный extend.

-- Метод: пронумеровать узлы в порядке обхода
numberNodes :: Tree a -> Tree (a, Int)
numberNodes t = fst (go 0 t)
  where
    go n (Node x ts) =
      let (ts', n') = foldr (\ ch (acc, counter) ->
                               let (ch', counter') = go counter ch
                               in  (ch':acc, counter'))
                             ([], n+1) ts
      in  (Node (x, n) ts', n')

putStrLn "Дерево с номерами узлов:"
putStr (showTree (numberNodes sampleTree))

Дерево с номерами узлов:

(1,0)
  (2,3)
    (4,5)
    (5,4)
  (3,1)
    (6,2)

In [35]:
-- Пример 4: Контекстно-зависимые преобразования дерева
-- "Отметить узлы, у которых все листья поддерева чётные"
allEvenLeaves :: Tree Int -> Bool
allEvenLeaves (Node x [])  = even x
allEvenLeaves (Node _ ts)  = all allEvenLeaves ts

markedTree :: Tree (Int, Bool)
markedTree = extend (\ t -> (extract t, allEvenLeaves t)) sampleTree

putStrLn "Узлы с признаком 'все листья чётные':"
putStr (showTree markedTree)

Узлы с признаком 'все листья чётные':

(1,False)
  (2,False)
    (4,True)
    (5,False)
  (3,True)
    (6,True)

## 💡 Ловушки

- **`duplicate` = дерево деревьев**: каждый узел заменяется поддеревом *с корнем в нём*.
  Это не «все возможные фокусы» (как в Zipper) — нет доступа к родителю или соседним ветвям!
- **Дерево смотрит только «вниз»**: `extend f` позволяет `f` видеть поддерево, но не надерево.
  Для доступа к «пути до корня» нужен другой Zipper (Tree Zipper / ContextZipper).
- **Rose Tree vs Binary Tree**: Rose Tree с произвольным числом детей — стандарт.
  Binary Tree — отдельный тип с другим инстансом.

## 🔭 Категорная точка зрения

`Tree` — это **свободная монада над функтором списка** (= монада на уровне наборов),
и одновременно **комонада над функтором-деревом**:
```
  T = νX. A × [X]    (коиндуктивное определение)
  ε (Node a _) = a
  δ t          = Node t (map δ (subForest t))
```
Структура дерева — это **несвободная коалгебра**: в отличие от Stream (терминальная),
дерево конечно и определяется через индуктивный тип.
Комонадная структура = «каждое поддерево — самостоятельный контекст».

---
# 9️⃣ CoFree f — «Свободная комонада»

## 📐 Реализация

`CoFree f a` — дуал к `Free f a`. Там где `Free` строит дерево из AST-узлов,
`CoFree` строит **бесконечное аннотированное дерево** (co-stream, co-tree).

```haskell
data CoFree f a = a :< f (CoFree f a)
--               метка    дети (в функторе f)
```

**Интуиция:**
- `Free f a` = дерево с листьями типа `a` и узлами типа `f`
- `CoFree f a` = дерево где **каждый узел** помечен `a` и имеет детей `f (CoFree f a)`

Это **обобщение**:
- `CoFree Identity` ≅ `Stream` (бесконечная цепочка)
- `CoFree []` ≅ `Tree` (розовое дерево)
- `CoFree Maybe` ≅ `Stream + конечность` (потенциально конечный поток)

In [36]:
-- CoFree как обобщение: CoFree f a = a :< f (CoFree f a)
data CoFree f a = a :< f (CoFree f a)

infixr 5 :<

instance Functor f => Functor (CoFree f) where
  fmap g (a :< fs) = g a :< fmap (fmap g) fs

instance Functor f => Comonad (CoFree f) where
  extract   (a :< _)  = a
  duplicate t@(_ :< fs) = t :< fmap duplicate fs

-- Вспомогательные
cohead :: CoFree f a -> a
cohead (a :< _) = a

cotail :: CoFree f a -> f (CoFree f a)
cotail (_ :< fs) = fs

-- Развернуть CoFree из коалгебры (аналог unfold для Stream)
unfoldCoFree :: Functor f => (s -> a) -> (s -> f s) -> s -> CoFree f a
unfoldCoFree label next s = label s :< fmap (unfoldCoFree label next) (next s)

## 🛠️ Примеры использования

In [37]:
-- Пример 1: CoFree Identity ≅ Stream
-- Identity = одно значение => бесконечная цепочка

newtype Identity' a = Identity' { runIdentity' :: a } deriving Functor

-- CoFree Identity a = a :< Identity (CoFree Identity a)
-- ≅ a :> CoFree Identity a  = Stream a

-- Создаём Stream натуральных чисел через CoFree Identity
natsCoFree :: CoFree Identity' Int
natsCoFree = unfoldCoFree id (Identity' . (+1)) 0

-- Взять n элементов
takeCoFree :: Functor f => Int -> CoFree f a -> [a]
takeCoFree 0 _        = []
takeCoFree n (a :< _) = a : case n of
  1 -> []
  _ -> takeCoFree (n-1) (runIdentity' (cotail (a :< undefined)))

-- Более корректно:
takeStream :: Int -> CoFree Identity' Int -> [Int]
takeStream 0 _         = []
takeStream n (a :< Identity' t) = a : takeStream (n-1) t

print $ takeStream 10 natsCoFree

[0,1,2,3,4,5,6,7,8,9]

In [38]:
-- Пример 2: CoFree [] ≅ Rose Tree с бесконечным разворачиванием
-- CoFree [] a = a :< [CoFree [] a]

-- Дерево чисел: каждый узел n имеет детей [n*2, n*2+1] (бинарное)
binaryCoTree :: CoFree [] Int
binaryCoTree = unfoldCoFree id (\ n -> [n*2, n*2+1]) 1

-- Взять уровень дерева
levelCoFree :: Int -> CoFree [] a -> [a]
levelCoFree 0 (a :< _)   = [a]
levelCoFree n (_ :< kids) = concatMap (levelCoFree (n-1)) kids

putStrLn "Уровни бинарного дерева (CoFree []):"
mapM_ (\ lv -> putStrLn $ "Уровень " ++ show lv ++ ": " ++ show (levelCoFree lv binaryCoTree))
      [0..3]

Уровни бинарного дерева (CoFree []):

Уровень 0: [1]
Уровень 1: [2,3]
Уровень 2: [4,5,6,7]
Уровень 3: [8,9,10,11,12,13,14,15]

In [39]:
-- Пример 3: CoFree Maybe — потенциально конечный поток
-- CoFree Maybe a = a :< Maybe (CoFree Maybe a)
-- Nothing = конец, Just t = продолжение

finiteStream :: CoFree Maybe Int
finiteStream = 1 :< Just (2 :< Just (3 :< Nothing))

-- extend: аннотировать длину оставшегося потока
remainingLength :: CoFree Maybe Int -> Int
remainingLength (_ :< Nothing) = 0
remainingLength (_ :< Just t)  = 1 + remainingLength t

annotated :: CoFree Maybe (Int, Int)
annotated = extend (\ t -> (cohead t, remainingLength t)) finiteStream

let toListM :: CoFree Maybe a -> [a]
    toListM (a :< Nothing) = [a]
    toListM (a :< Just t)  = a : toListM t

putStrLn "CoFree Maybe: поток с длиной оставшейся части:"
print $ toListM annotated

CoFree Maybe: поток с длиной оставшейся части:

[(1,2),(2,1),(3,0)]

## 💡 Ловушки

- **`CoFree f` бесконечна** при `f = Identity` или `f = []` с непустыми детьми.
  Это нормально в Haskell благодаря лени, но требует осторожности при форсировании.
- **Разница с `Free`**: `Free f` = **индуктивное** дерево (конечное, с листьями),
  `CoFree f` = **коиндуктивное** дерево (потенциально бесконечное, метки в каждом узле).
- **`extract` = метка корня**, `duplicate` = «каждый узел заменяется поддеревом».
  Это точно как в `Tree`, но обобщённо для любого функтора `f`.

## 🔭 Категорная точка зрения

`CoFree f` — **правый сопряжённый к забывающему функтору** из комонад в функторы:
```
  Comonad(W, F)  ≅  Functor(W, CoFree F)
```
Это дуал к `Free`:
```
  Free F  = левый сопряжённый к Forget : Monad → Functor
  CoFree F = правый сопряжённый к Forget : Comonad → Functor
```
```
                    Forget
  Comonad  ──────────────→  Functor
     ↑   ←──────────────      ↑
     │        CoFree           │
  CoFree F   =  наименьшая комонада над F
```
Это означает: любая коалгебра над `F` однозначно факторизуется через `CoFree F`.

---
# 🔟 Representable — «Представимые функторы»

## 📐 Реализация

Функтор `f` **представим** (Representable), если он изоморфен `(->) r` для некоторого `r`:
```
  f ≅ (r →)    ⟺  f a ≅ r -> a
```

```haskell
class Functor f => Representable f where
  type Rep f    -- тип "индексов"
  index  :: f a -> Rep f -> a   -- f a ≅ Rep f -> a (читать)
  tabulate :: (Rep f -> a) -> f a  -- построить из функции
```

**Интуиция:** любой представимый функтор — это «массив, проиндексированный `Rep f`».
- `index` = доступ по индексу
- `tabulate` = создать массив по формуле

**Связь с `Store`**: `Store (Rep f) a ≅ f a` — представимые функторы — это ровно те,
для которых существует комонада `Store (Rep f)`.

In [40]:
:set -XTypeFamilies

-- Класс Representable
class Functor f => Representable f where
  type Rep f
  index    :: f a -> Rep f -> a
  tabulate :: (Rep f -> a) -> f a

-- Закон: tabulate . index = id,  index . tabulate = id

-- Пример 1: Pair — представим Bool
data Pair a = Pair a a deriving (Show, Functor)

instance Representable Pair where
  type Rep Pair = Bool
  index (Pair x _) False = x
  index (Pair _ y) True  = y
  tabulate f = Pair (f False) (f True)

In [41]:
-- Пример 2: Vec2 — представим (Bool, Bool) = 4 позиции
data Vec2 a = Vec2 a a a a deriving (Show, Functor)

instance Representable Vec2 where
  type Rep Vec2 = (Bool, Bool)
  index (Vec2 a _ _ _) (False, False) = a
  index (Vec2 _ b _ _) (False, True)  = b
  index (Vec2 _ _ c _) (True,  False) = c
  index (Vec2 _ _ _ d) (True,  True)  = d
  tabulate f = Vec2 (f (False,False)) (f (False,True))
                    (f (True,False))  (f (True,True))

-- Операции через Store:
-- extract = index arr (текущий индекс)
-- duplicate arr = tabulate (\ i -> tabulate (\ j -> index arr j)) -- с фокусом i

In [42]:
-- Пример 3: Комонада для Representable (через Store)
-- Любой Representable порождает комонаду!

data RepStore f a = RepStore (f a) (Rep f)

extractRep :: Representable f => RepStore f a -> a
extractRep (RepStore fa i) = index fa i

duplicateRep :: Representable f => RepStore f a -> RepStore f (RepStore f a)
duplicateRep (RepStore fa i) = RepStore (tabulate (\ j -> RepStore fa j)) i

-- Демо с Pair
pairStore :: RepStore Pair Int
pairStore = RepStore (Pair 10 20) False  -- фокус на False (первый элемент)

putStrLn $ "extract: " ++ show (extractRep pairStore)  -- 10

let dup = duplicateRep pairStore
putStrLn $ "duplicate → extract . extract: " ++ show (extractRep (extractRep dup))  -- 10

extract: 10

duplicate → extract . extract: 10

In [43]:
-- Пример 4: Функциональный массив как Representable
-- Pair как простой массив 2 элементов
mapPair :: (a -> b) -> Pair a -> Pair b
mapPair f (Pair x y) = Pair (f x) (f y)

-- "Произведение" двух Pair
zipPair :: Pair a -> Pair b -> Pair (a,b)
zipPair (Pair a b) (Pair c d) = Pair (a,c) (b,d)

p1 :: Pair Int
p1 = Pair 3 7

-- tabulate создаёт по функции
p2 :: Pair Int
p2 = tabulate (\ b -> if b then 100 else 200)

putStrLn $ "p1 = " ++ show p1
putStrLn $ "p2 = " ++ show p2
putStrLn $ "zipPair = " ++ show (zipPair p1 p2)

-- Representable автоматически даёт Applicative и Monad!
-- pure x = tabulate (const x)
-- (f <*> x) i = (index f i) (index x i)
pureP :: a -> Pair a
pureP x = tabulate (const x)

putStrLn $ "pure 42 = " ++ show (pureP (42 :: Int))

p1 = Pair 3 7

p2 = Pair 200 100

zipPair = Pair (3,200) (7,100)

pure 42 = Pair 42 42

## 💡 Ловушки

- **Не все функторы представимы**: `Maybe`, `[]`, `Either e` — нет (разный «размер»).
  `Pair`, `Vec n`, `(->) r` — да.
- **`Rep f` должен быть изоморфизмом**: `index` и `tabulate` — взаимно обратны.
  Если нарушить закон, операции ведут себя непредсказуемо.
- **Representable ⟹ Applicative**: `pure = tabulate . const`,
  `f <*> x = tabulate (\i -> index f i (index x i))`. Это бесплатно.
- **Representable ⟹ Distributive**: `distribute = tabulate . flip (fmap . flip index)`.

## 🔭 Категорная точка зрения

Представимые функторы — это **функторы-образы объектов** через Yoneda:
```
  Hom(r, ─) : Set → Set    (функтор Йонеды)
  f ≅ Hom(r, ─)  ⟺  f представим с Rep = r
```
Лемма Йонеды: `Nat(Hom(r,─), f) ≅ f r`
Это значит: **натуральные преобразования из Hom(r,─) = элементы f r**.
Отсюда изоморфизм `f a ≅ r → a` для представимых `f`.

Связь с комонадами:
```
  f представим ⟺ существует сопряжение Δ ⊣ f (диагональ ⊣ f)
  ⟺ f ∘ Δ = Store r  (комонада!)
```

---
# 🪞 Дуальность: Монада ↔ Комонада

## 11.1 Таблица операций

| | **Монада `m`** | **Комонада `w`** |
|-|----------------|-----------------|
| Тип-конструктор | `m :: * -> *` | `w :: * -> *` |
| Вложение / Извлечение | `return :: a -> m a` | `extract :: w a -> a` |
| Склейка / Расширение | `join :: m (m a) -> m a` | `duplicate :: w a -> w (w a)` |
| Связывание | `(>>=) :: m a -> (a -> m b) -> m b` | `extend :: (w a -> b) -> w a -> w b` |
| Рыбий оператор | `(>=>) :: (a->m b)->(b->m c)->a->m c` | `(=<=) :: (w b->c)->(w a->b)->w a->c` |
| Закон единицы | `return >=> f = f` | `extract . extend f = f` |
| Закон единицы 2 | `f >=> return = f` | `extend extract = id` |
| Ассоциативность | `(f>=>g)>=>h = f>=>(g>=>h)` | `extend f . extend g = extend (f . extend g)` |

---

## 11.2 Операции на уровне типов (стрелки обращены)

```
  МОНАДА                          КОМОНАДА
  ─────────────────────────────   ──────────────────────────────────
  η : Id → M    (return)          ε : W → Id    (extract)
  μ : M∘M → M   (join)            δ : W → W∘W   (duplicate)

  Диаграммы коммутативности:

  МОНАДА (законы μ и η):
  ┌──────────────────────┐   ┌──────────────────────┐
  │  M∘M∘M   μ∘id→  M∘M │   │  M    η∘id→   M∘M    │
  │   id∘μ↓       μ↓    │   │  ‖    id∘η↓    μ↓    │
  │   M∘M   ──μ──→  M   │   │  M   ←──μ───   M∘M   │
  └──────────────────────┘   └──────────────────────┘

  КОМОНАДА (законы δ и ε):
  ┌──────────────────────┐   ┌──────────────────────┐
  │  W∘W∘W  ←δ∘id─ W∘W  │   │  W    ←ε∘id─  W∘W   │
  │   id∘δ↑       δ↑    │   │  ‖    id∘ε↑    δ↑   │
  │   W∘W   ←─δ──  W    │   │  W   ──δ───→   W∘W   │
  └──────────────────────┘   └──────────────────────┘

  Всё то же самое, но стрелки ОБРАЩЕНЫ!
```

In [44]:
-- Демо дуальности: законы комонады
-- Проверим на Zipper (конечный, с остановкой на краях)

data Zipper' a = Zipper' [a] a [a] deriving (Show, Eq)

instance Functor Zipper' where
  fmap f (Zipper' l c r) = Zipper' (map f l) (f c) (map f r)

goLeft' :: Zipper' a -> Zipper' a
goLeft'  (Zipper' (l:ls) c r) = Zipper' ls l (c:r)
goLeft'  z                    = z

goRight' :: Zipper' a -> Zipper' a
goRight' (Zipper' l c (r:rs)) = Zipper' (c:l) r rs
goRight' z                    = z

instance Comonad Zipper' where
  extract (Zipper' _ c _) = c
  duplicate z@(Zipper' l c r) = Zipper' lefts z rights
    where
      lefts  = unfoldr goL z
      rights = unfoldr goR z
      goL (Zipper' [] _ _)    = Nothing
      goL zz = let zz' = goLeft' zz  in Just (zz', zz')
      goR (Zipper' _ _ [])    = Nothing
      goR zz = let zz' = goRight' zz in Just (zz', zz')

-- Тест на конечном Zipper
z :: Zipper' Int
z = Zipper' [2,1] 3 [4,5]

-- Закон 1: extract . duplicate = id
law1Z :: Bool
law1Z = (extract . duplicate) z == z

-- Закон 2: fmap extract . duplicate = id
law2Z :: Bool
law2Z = (fmap extract . duplicate) z == z

-- Закон 3: проверяем частично через extract
law3ZL = (extract . extract . duplicate . duplicate) z
law3ZR = (extract . extract . fmap duplicate . duplicate) z

putStrLn $ "Закон 1 (extract∘duplicate=id): " ++ show law1Z
putStrLn $ "Закон 2 (fmap extract∘duplicate=id): " ++ show law2Z
putStrLn $ "Закон 3 (оба extract дают то же): " ++ show (law3ZL == law3ZR)

Закон 1 (extract∘duplicate=id): True

Закон 2 (fmap extract∘duplicate=id): True

Закон 3 (оба extract дают то же): True

## 11.3 Категорный взгляд: сопряжения (Adjunctions)

Каждая пара (монада, комонада) возникает из **сопряжения функторов**.

```
Сопряжение:   F ⊣ G    (F : C → D,  G : D → C)
              означает: Hom_D(F a, b) ≅ Hom_C(a, G b)

Монада:   T = G ∘ F : C → C
          η : Id → GF       (единица сопряжения → return)
          μ : GFGF → GF     (через ε: GεF → id)

Комонада: S = F ∘ G : D → D
          ε : FG → Id       (коединица сопряжения → extract)
          δ : FG → FGFG     (через η: FηG)
```

### Главные сопряжения в Haskell

```
Сопряжение         Монада (GF)           Комонада (FG)
───────────────────────────────────────────────────────
(─×e) ⊣ (e→─)     State e  ≅ GF        Store e  ≅ FG
Δ ⊣ Hom(r,─)      Reader r ≅ r→─        Env r    ≅ (r,─)
Free ⊣ Forget      List/Free monad       CoFree
Id ⊣ Id            Identity monad        Identity comonad
```

```
Детали (─×e) ⊣ (e→─):

  F = (─×e) : Set → Set      (умножить на e)
  G = (e→─) : Set → Set      (функции из e)

  Монада GF:  a ↦ e→(a×e) ≅ e→a×e = State e a     ✓
  Комонада FG: a ↦ (e→a)×e = Store e a              ✓

  η_a : a → e→(a×e)           return x = \e -> (x, e)
  ε_a : (e→a)×e → a           extract (f, e) = f e
```

In [45]:
-- Демо: сопряжение (─×e) ⊣ (e→─) даёт State и Store

-- Единица сопряжения (= return монады State):
unit :: a -> e -> (a, e)
unit x e = (x, e)

-- Коединица сопряжения (= extract комонады Store):
counit :: (e -> a, e) -> a
counit (f, e) = f e

-- Из сопряжения получаем return State:
returnState :: a -> (e -> (a, e))
returnState = unit

-- Из сопряжения получаем extract Store:
extractStore :: (e -> a, e) -> a
extractStore = counit

demonstration :: String
demonstration = unlines
  [ "unit    = return State (unit x e = (x,e))"
  , "counit  = extract Store (counit (f,e) = f e)"
  , "Монада  = G∘F : a ↦ e→(a,e) = State e a"
  , "Комонада= F∘G : a ↦ (e→a,e) = Store e a"
  , "curry   : (a,e)→b  ≅  a→(e→b)  — транспозиция сопряжения (bijection)"
  ]
putStr demonstration

unit    = return State (unit x e = (x,e))
counit  = extract Store (counit (f,e) = f e)
Монада  = G∘F : a ↦ e→(a,e) = State e a
Комонада= F∘G : a ↦ (e→a,e) = Store e a
curry   : (a,e)→b  ≅  a→(e→b)  — транспозиция сопряжения (bijection)

## 11.4 Категория Клейсли vs КоКлейсли

```
Категория Клейсли (для монады M):
  Объекты: те же что в C
  Морфизмы: a → M b   (a ─Kleisli→ b)
  Композиция: f >=> g = \x -> f x >>= g
  Тождество: return

Категория коКлейсли (для комонады W):
  Объекты: те же что в C
  Морфизмы: W a → b   (a ─coKleisli→ b)
  Композиция: f =<= g = f . extend g
  Тождество: extract
```

```
  В Клейсли:   a ──f──→ M b ──g──→ M c
  В коКлейсли: W a ←──── b ←──── W c
                   f           g
  (стрелки «обращены» относительно исходной категории!)
```

Комонадические «эффекты» = **вычисление зависит от контекста**, а не строит его.
Монадические «эффекты» = **вычисление строит контекст**, а не зависит от него.

In [46]:
-- Демо: категория коКлейсли на Zipper'
-- Морфизмы W a -> b, композиция через extend

-- CoKleisli морфизм: посмотреть на соседей
leftNeighbor :: Zipper' Int -> Maybe Int
leftNeighbor (Zipper' []    _ _) = Nothing
leftNeighbor (Zipper' (l:_) _ _) = Just l

rightNeighbor :: Zipper' Int -> Maybe Int
rightNeighbor (Zipper' _ _ []   ) = Nothing
rightNeighbor (Zipper' _ _ (r:_)) = Just r

sumNeighbors :: Zipper' Int -> Int
sumNeighbors z =
  let l = maybe 0 id (leftNeighbor z)
      r = maybe 0 id (rightNeighbor z)
  in  l + r

-- Категория коКлейсли: extract = тождественный морфизм
identityMorphism :: Zipper' Int -> Int
identityMorphism = extract

-- Проверка: extract . extend f = f (закон 1)
testLaw :: Bool
testLaw = let f = sumNeighbors
              z = Zipper' [2,1] 3 [4,5]
          in  (extract . extend f) z == f z

-- Композиция в коКлейсли: f =<= g = f . extend g
(=<=) :: Comonad w => (w b -> c) -> (w a -> b) -> w a -> c
f =<= g = f . extend g

-- Сначала суммируем соседей, потом берём extract из результата
composed :: Zipper' Int -> Int
composed = identityMorphism =<= sumNeighbors

putStrLn $ "Закон 1 (extract∘extend f = f): " ++ show testLaw
putStrLn $ "composed z = " ++ show (composed (Zipper' [2,1] 3 [4,5]))

Line 15: Use fromMaybe
Found:
maybe 0 id
Why not:
Data.Maybe.fromMaybe 0Line 16: Use fromMaybe
Found:
maybe 0 id
Why not:
Data.Maybe.fromMaybe 0

Закон 1 (extract∘extend f = f): True

composed z = 6

## 11.5 Конкретные дуальные пары

![Dual&#39;nye pary](../diagrams/comonads/cm_comonad_pairs.svg)

In [47]:
-- Демо: почему Maybe не комонада
-- extract :: Maybe a -> a  — невозможно написать тотально!

-- Это НЕ работает:
-- extractMaybe :: Maybe a -> a
-- extractMaybe (Just x) = x
-- extractMaybe Nothing  = ???  -- нет значения!

-- Сравнение: Stream, NonEmpty, Store — всегда имеют "текущее значение"
-- Maybe, Either, [] — нет гарантии

safeExtract :: Maybe Int -> String
safeExtract (Just x) = "Есть: " ++ show x
safeExtract Nothing  = "Нет значения — extract невозможен!"

putStrLn $ safeExtract (Just 42)
putStrLn $ safeExtract Nothing
putStrLn "Вывод: Maybe не может быть комонадой (extract нетотален)"

Есть: 42

Нет значения — extract невозможен!

Вывод: Maybe не может быть комонадой (extract нетотален)

## 11.6 Итоговая диаграмма

![Иерархия комонад](../diagrams/comonads/cm_final.svg)

In [48]:
-- Финальное демо: одна задача, два подхода
-- Задача: "сгладить сигнал" (скользящее среднее по 3 элементам)

-- МОНАДИЧЕСКИЙ подход (State): явная передача индекса
import Control.Monad.State

smoothMonadic :: [Double] -> [Double]
smoothMonadic xs = evalState (mapM step [0..n-1]) xs
  where
    n = length xs
    step i = do
      arr <- get
      let idxs      = [max 0 (i-1)..min (n-1) (i+1)]
          neighbors = map (arr !!) idxs
      return (sum neighbors / fromIntegral (length neighbors))

-- КОМОНАДИЧЕСКИЙ подход: extend на Zipper
-- Определяем локальный Zipper прямо здесь, чтобы не конфликтовать
data ZipD = ZipD [Double] Double [Double]

zipFromList :: [Double] -> Maybe ZipD
zipFromList []     = Nothing
zipFromList (x:xs) = Just (ZipD [] x xs)

zipToList :: ZipD -> [Double]
zipToList (ZipD ls c rs) = reverse ls ++ [c] ++ rs

zipGoR :: ZipD -> ZipD
zipGoR z@(ZipD l c [])     = z
zipGoR   (ZipD l c (r:rs)) = ZipD (c:l) r rs

zipGoL :: ZipD -> ZipD
zipGoL z@(ZipD [] c r)     = z
zipGoL   (ZipD (l:ls) c r) = ZipD ls l (c:r)

-- extend: применить функцию ко всем позициям
extendZip :: (ZipD -> Double) -> ZipD -> ZipD
extendZip f z@(ZipD ls c rs) = ZipD ls' (f z) rs'
  where
    ls' = map f (take (length ls) (tail (iterate zipGoL z)))
    rs' = map f (take (length rs) (tail (iterate zipGoR z)))

avg3 :: ZipD -> Double
avg3 (ZipD ls c rs) =
  let neighbors = take 1 ls ++ [c] ++ take 1 rs
  in  sum neighbors / fromIntegral (length neighbors)

smoothComonadic :: [Double] -> [Double]
smoothComonadic xs = case zipFromList xs of
  Nothing -> []
  Just z  -> zipToList (extendZip avg3 z)

-- Демо
signal' :: [Double]
signal' = [1.0, 3.0, 5.0, 3.0, 1.0, 2.0, 4.0, 2.0, 1.0]

putStrLn "Исходный сигнал:"
print signal'
putStrLn "Монадический подход (State):"
print (smoothMonadic signal')
putStrLn "Комонадический подход (Zipper+extend):"
print (smoothComonadic signal')
putStrLn ""
putStrLn "Ключевое различие:"
putStrLn "  State: явный индекс arr!!, get/put — императивный стиль"
putStrLn "  Zipper+extend: avg3 видит весь контекст через структуру данных"

Исходный сигнал:

[1.0,3.0,5.0,3.0,1.0,2.0,4.0,2.0,1.0]

Монадический подход (State):

[2.0,3.0,3.6666666666666665,3.0,2.0,2.3333333333333335,2.6666666666666665,2.3333333333333335,1.5]

Комонадический подход (Zipper+extend):

[2.0,3.0,3.6666666666666665,3.0,2.0,2.3333333333333335,2.6666666666666665,2.3333333333333335,1.5]

Ключевое различие:

  State: явный индекс arr!!, get/put — императивный стиль

  Zipper+extend: avg3 видит весь контекст через структуру данных

## Диаграмма: монада vs комонада

Двойственность return/extract, bind/extend.

![Монада vs Комонада](../diagrams/comonads/cm_comonad.svg)

---

**← Предыдущий:** [Трансформеры монад](MonadTransformers.ipynb)  |  **Следующий →** [Трансформеры комонад](ComonadTransformers.ipynb)
